# Mini-FORESIGHT — Inventory Risk Analysis

Forecasting tells us **expected future demand**.

Inventory risk analysis combines **expected demand** with **available stock** and **supplier lead time** to identify products that may run out of stock.

This notebook will calculate:
- Current stock
- Average daily demand
- Forecast demand
- Lead-time demand
- Days of stock coverage
- Stockout risk
- Risk level

The final result will identify which SKUs require inventory attention.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")

In [ ]:
inventory = pd.read_csv(
    "../data/processed/inventory_snapshots_clean.csv"
)

sku_master = pd.read_csv(
    "../data/processed/sku_master_clean.csv"
)

forecast = pd.read_csv(
    "../data/processed/forecast_results.csv"
)

inventory["date"] = pd.to_datetime(inventory["date"])
forecast["date"] = pd.to_datetime(forecast["date"])

inventory = inventory.sort_values(["sku_id", "date"])

print("=== Dataset Shapes ===")
print("Inventory:", inventory.shape)
print("SKU master:", sku_master.shape)
print("Forecast:", forecast.shape)

## Current Inventory

The **latest inventory snapshot** is used as the current inventory position.

For each SKU, we take the `closing_stock` from the latest available date, which is **2025-01-14**.

We do **NOT** use the historical minimum stock as the current stock.

The latest closing stock is the correct starting point for the risk calculation.

In [ ]:
latest_inventory = (
    inventory
    .sort_values(["sku_id", "date"])
    .groupby("sku_id")
    .tail(1)
    [["sku_id", "date", "closing_stock"]]
    .rename(columns={
        "date": "inventory_date",
        "closing_stock": "current_stock"
    })
    .reset_index(drop=True)
)

print("=== Current Inventory ===")
print(latest_inventory.to_string(index=False))

## Historical Average Daily Demand

Historical average daily demand is calculated from the **cleaned sales dataset**.

This gives us a simple estimate of **normal daily demand** for each SKU.

It will be used as an additional inventory-risk indicator.

We do **NOT** use future forecast values to calculate historical average demand.

In [ ]:
sales = pd.read_csv(
    "../data/processed/sales_daily_clean.csv"
)

sales["date"] = pd.to_datetime(sales["date"])

avg_daily_demand = (
    sales.groupby("sku_id")["units_sold"]
    .mean()
    .reset_index()
    .rename(columns={
        "units_sold": "avg_daily_demand"
    })
)

print("=== Historical Average Daily Demand ===")
avg_daily_demand_display = avg_daily_demand.copy()
avg_daily_demand_display["avg_daily_demand"] = avg_daily_demand_display["avg_daily_demand"].round(2)
print(avg_daily_demand_display.to_string(index=False))

## Forecast Demand

The **3-day machine-learning forecast** from Step 6 is used to estimate near-term demand.

For each SKU we calculate:
- **Total forecast demand** over the next 3 days
- **Average forecast demand** per day

These values help us understand short-term inventory pressure.

In [ ]:
forecast_summary = (
    forecast.groupby("sku_id")["forecast_units"]
    .agg(
        forecast_3_day_demand="sum",
        forecast_avg_daily_demand="mean"
    )
    .reset_index()
)

print("=== 3-Day Forecast Demand ===")
forecast_summary_display = forecast_summary.copy()
forecast_summary_display["forecast_3_day_demand"] = forecast_summary_display["forecast_3_day_demand"].round(2)
forecast_summary_display["forecast_avg_daily_demand"] = forecast_summary_display["forecast_avg_daily_demand"].round(2)
print(forecast_summary_display.to_string(index=False))

## Lead-Time Demand

**Lead time** is the number of days required for a supplier to deliver replenishment stock.

If a product has a lead time of 7 days and normally sells about 2 units per day, approximately:

```
7 × 2 = 14 units
```

may be required during the supplier lead time. This is called **lead-time demand**.

For this project, we calculate:

```
lead_time_demand = avg_daily_demand × lead_time_days
```

We use **historical average daily demand** for this calculation because the available ML forecast covers only 3 days.

In [ ]:
risk = latest_inventory.merge(
    avg_daily_demand,
    on="sku_id",
    how="left"
)

risk = risk.merge(
    sku_master[
        ["sku_id", "product_name", "category",
         "unit_price", "lead_time_days"]
    ],
    on="sku_id",
    how="left"
)

risk = risk.merge(
    forecast_summary,
    on="sku_id",
    how="left"
)

risk["lead_time_demand"] = (
    risk["avg_daily_demand"]
    * risk["lead_time_days"]
)

print("=== Inventory Risk Base Table ===")
base_display = risk[[
    "sku_id", "product_name", "current_stock",
    "avg_daily_demand", "lead_time_days",
    "lead_time_demand", "forecast_3_day_demand"
]].copy()
base_display["avg_daily_demand"] = base_display["avg_daily_demand"].round(2)
base_display["lead_time_demand"] = base_display["lead_time_demand"].round(2)
base_display["forecast_3_day_demand"] = base_display["forecast_3_day_demand"].round(2)
print(base_display.to_string(index=False))

## Days of Stock Coverage

**Days of stock coverage** estimates how many days the current inventory could support demand.

Formula:

```
days_of_stock = current_stock / avg_daily_demand
```

- A **higher** value means more inventory coverage.
- A **lower** value means greater inventory pressure.

This is an approximate indicator and assumes demand remains near the historical average.

In [ ]:
risk["days_of_stock"] = (
    risk["current_stock"]
    / risk["avg_daily_demand"]
)

print("=== Stock Coverage ===")
coverage_display = risk[[
    "sku_id", "current_stock", "avg_daily_demand", "days_of_stock"
]].copy()
coverage_display["avg_daily_demand"] = coverage_display["avg_daily_demand"].round(2)
coverage_display["days_of_stock"] = coverage_display["days_of_stock"].round(2)
print(coverage_display.to_string(index=False))

## Stockout Risk Logic

We use a simple and explainable rule.

Risk considers whether current stock can cover **supplier lead-time demand**.

We calculate:

```
stock_surplus_vs_lead_time = current_stock - lead_time_demand
```

Interpretation:
- If current stock is **below** lead-time demand → **High risk**
- If current stock is **close to** lead-time demand → **Medium risk**
- If current stock **comfortably exceeds** lead-time demand → **Low risk**

Thresholds:

```
HIGH:   current_stock < lead_time_demand
MEDIUM: lead_time_demand <= current_stock <= lead_time_demand * 1.5
LOW:    current_stock > lead_time_demand * 1.5
```

This is a simple business rule for this project, not a universal inventory-management standard.

In [ ]:
risk["stock_surplus_vs_lead_time"] = (
    risk["current_stock"]
    - risk["lead_time_demand"]
)

def classify_risk(row):
    if row["current_stock"] < row["lead_time_demand"]:
        return "High"
    elif row["current_stock"] <= row["lead_time_demand"] * 1.5:
        return "Medium"
    else:
        return "Low"

risk["risk_level"] = risk.apply(
    classify_risk,
    axis=1
)

print("=== Risk Classification ===")
risk_class_display = risk[[
    "sku_id", "product_name", "current_stock",
    "lead_time_demand", "days_of_stock", "risk_level"
]].copy()
risk_class_display["lead_time_demand"] = risk_class_display["lead_time_demand"].round(2)
risk_class_display["days_of_stock"] = risk_class_display["days_of_stock"].round(2)
print(risk_class_display.to_string(index=False))

## Forecast vs Inventory

The next 3 days of forecast demand can be compared with current inventory.

We calculate the remaining stock after consuming the full 3-day forecast:

```
projected_stock_after_3_days = current_stock - forecast_3_day_demand
```

This is **NOT** a reorder recommendation.

It is only a short-term stock pressure indicator.

In [ ]:
risk["projected_stock_after_3_days"] = (
    risk["current_stock"]
    - risk["forecast_3_day_demand"]
)

print("=== Projected Stock After 3 Forecast Days ===")
projected_display = risk[[
    "sku_id", "current_stock", "forecast_3_day_demand",
    "projected_stock_after_3_days", "risk_level"
]].copy()
projected_display["forecast_3_day_demand"] = projected_display["forecast_3_day_demand"].round(2)
projected_display["projected_stock_after_3_days"] = projected_display["projected_stock_after_3_days"].round(2)
print(projected_display.to_string(index=False))

In [ ]:
risk_final = risk[[
    "sku_id",
    "product_name",
    "category",
    "current_stock",
    "avg_daily_demand",
    "forecast_avg_daily_demand",
    "forecast_3_day_demand",
    "lead_time_days",
    "lead_time_demand",
    "days_of_stock",
    "stock_surplus_vs_lead_time",
    "projected_stock_after_3_days",
    "risk_level"
]].copy()

risk_order = {"High": 0, "Medium": 1, "Low": 2}
risk_final["_order"] = risk_final["risk_level"].map(risk_order)
risk_final = risk_final.sort_values(["_order", "days_of_stock"]).drop(columns="_order").reset_index(drop=True)

print("=== FINAL INVENTORY RISK TABLE ===")
risk_final_display = risk_final.copy()
for col in ["avg_daily_demand", "forecast_avg_daily_demand", "forecast_3_day_demand",
            "lead_time_demand", "days_of_stock", "stock_surplus_vs_lead_time",
            "projected_stock_after_3_days"]:
    risk_final_display[col] = risk_final_display[col].round(2)
print(risk_final_display.to_string(index=False))

## Risk Interpretation

The analysis identifies:
- Which SKU has the **lowest stock coverage**.
- Which SKU has the **highest lead-time demand**.
- Which SKU has the **greatest inventory pressure**.
- Which SKUs are classified as **High / Medium / Low** risk.

The observations below are generated **automatically from the actual calculated values** — nothing is hardcoded.

In [ ]:
print("=== Inventory Risk Observations ===")
print()

lowest_coverage_sku = risk_final.loc[risk_final["days_of_stock"].idxmin(), "sku_id"]
lowest_coverage_val = risk_final["days_of_stock"].min()
print("1. SKU with lowest stock coverage:")
print("   ", lowest_coverage_sku, "({:.2f} days)".format(lowest_coverage_val))
print()

highest_leadtime_sku = risk_final.loc[risk_final["lead_time_demand"].idxmax(), "sku_id"]
highest_leadtime_val = risk_final["lead_time_demand"].max()
print("2. SKU with highest lead-time demand:")
print("   ", highest_leadtime_sku, "({:.2f} units)".format(highest_leadtime_val))
print()

high_risk = risk_final[risk_final["risk_level"] == "High"]["sku_id"].tolist()
print("3. Highest-risk SKUs:")
print("   ", high_risk)
print()

low_risk = risk_final[risk_final["risk_level"] == "Low"]["sku_id"].tolist()
print("4. Lowest-risk SKUs:")
print("   ", low_risk if low_risk else "None")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(risk_final["sku_id"], risk_final["days_of_stock"])
plt.title("Inventory Coverage by SKU")
plt.xlabel("SKU")
plt.ylabel("Days of Stock Coverage")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

x = np.arange(len(risk_final))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, risk_final["current_stock"], width, label="Current Stock")
plt.bar(x + width/2, risk_final["lead_time_demand"], width, label="Lead-Time Demand")
plt.title("Current Stock vs Lead-Time Demand")
plt.xlabel("SKU")
plt.ylabel("Units")
plt.xticks(x, risk_final["sku_id"])
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
output_path = Path(
    "../data/processed/inventory_risk.csv"
)

risk_final.to_csv(
    output_path,
    index=False
)

print("Inventory risk results saved successfully:")
print(output_path)

In [ ]:
verification = pd.read_csv(
    "../data/processed/inventory_risk.csv"
)

print("=== Saved Inventory Risk Verification ===")
print("Shape:", verification.shape)
print()
print("Columns:", verification.columns.tolist())
print()
print("Missing values:")
print(verification.isna().sum())
print()
print("Data:")
print(verification.to_string(index=False))
print()
print("Expected 3 SKU rows:", len(verification) == 3)
print("Zero missing values:", verification.isna().sum().sum() == 0)
print("Risk level populated for every SKU:", verification["risk_level"].notna().all())

# Inventory Risk Summary

The inventory-risk stage combines:
- Current inventory
- Historical demand
- Forecast demand
- Supplier lead time

to estimate **inventory pressure**.

The analysis does **not** yet recommend how much to reorder.

It only identifies the **level of risk**.

The output file is:

```
data/processed/inventory_risk.csv
```

The next pipeline step is:

---

## Step 9 — Inventory Recommendations

That step will use the forecast and inventory-risk results to calculate practical reorder quantities and timing.